# Airbnb Price Category Classifier

In [ ]:
import pandas as pd
import numpy as np
import os 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras as keras
from sklearn.preprocessing import StandardScaler
import time

In [ ]:
# file paths for dataset
airbnb_filename = os.path.join(os.getcwd(), "airbnbListingsData.csv")

# load dataset and save it to df
df = pd.read_csv(airbnb_filename)

df.head()

In [ ]:
print(df.describe())

In [ ]:
nan_count = np.sum(df.isnull(), axis = 0)
nan_count

In [ ]:
# visualizations
sns.lineplot(
    data=df,
    x='accommodates',
    y=(df['price_category'] == 'high').astype(int)
)

plt.title('High Price by Accommodates')
plt.xlabel('accommodates')
plt.ylabel('high price')
plt.show()


In [ ]:
sns.countplot(data=df, x='price_category')
plt.title("Showing class imbalance between low and high priced Airbnb")

In [ ]:
# data cleaning
# 1. to_drop
to_drop = ['name', 'description', 'neighborhood_overview', 'host_location', 'host_about',
          'host_name', 'amenities']

df.drop(to_drop, axis=1, inplace=True)

In [ ]:
# 2. to_impute
to_impute = ['host_response_rate', 'host_acceptance_rate', 'bedrooms', 'beds']

for colname in to_impute:
    df[colname + '_na'] = df[colname].isnull()
    
df.columns

In [ ]:
# mean imputation
for colname in to_impute:
    df[colname] = df[colname].fillna(np.mean(df[colname]))

for colname in to_impute:
    print("{} missing values count :{}".format(colname, np.sum(df[colname].isnull(), axis = 0)))

In [ ]:
# see potential cols to one hot encode
print(df.select_dtypes(include=['object']).columns)

In [ ]:
# to_encode
to_encode = ['neighbourhood_group_cleansed', 'room_type']
df = pd.get_dummies(df, columns=to_encode, drop_first=True)

df.columns

In [ ]:
# encoding t/f cols
tf_cols = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability',
    'instant_bookable'
]

df = pd.get_dummies(df, columns=tf_cols, drop_first=True)


In [ ]:
# encoding price
# 1 = high price
# 0 = low price
df['price_category_binary'] = (df['price_category'] == 'high').astype(int)

print(df['price_category_binary'].value_counts())
print(df[['price_category', 'price_category_binary']].head())

In [ ]:
# Create labeled examples from the dataset
X = df.drop(columns=['price_category', 'price_category_binary', 'price'])
y = df['price_category_binary']

In [ ]:
# Create training and test sets out of the labeled examples 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1234)

In [ ]:
# Train, test and evaluate model

# create model object
log_reg_model = LogisticRegression(max_iter=5000)

# fit to training data
log_reg_model.fit(X_train, y_train)

# predictions on test data
log_reg_predictions = log_reg_model.predict(X_test)

# log reg accuracy score
log_reg_accuracy = accuracy_score(y_test, log_reg_predictions)

# log reg f1 score
log_reg_f1 = f1_score(y_test, log_reg_predictions, average='binary')

# print the results
print('Accuracy for LogReg model: ', log_reg_accuracy)
print('F1 Score for LogReg model: ', log_reg_f1)

In [ ]:
# Perform model selection through grid search cross-validation (GridSearchCV)
# to identify optimal hyperparameter values for model

# scale X_train and X_test 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# scale X_train and X_test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# param_grid for grid search parameter
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# Grid Search
grid = GridSearchCV(estimator=LogisticRegression(max_iter=5000), 
                    param_grid=param_grid, 
                    scoring='f1', 
                    cv=5)

# fit on scaled data
grid_search = grid.fit(X_train_scaled, y_train)

#print results
print('Best hyperparameters:', grid_search.best_params_)
print('Best cv F1 score:', grid_search.best_score_)

In [ ]:
# Train, test and evaluate a final version of model using the optimal hyperparameter values.

log_reg_final_model = grid_search.best_estimator_
log_reg_final_predict = log_reg_final_model.predict(X_test_scaled)

# evaluation
log_reg_final_accuracy = accuracy_score(y_test, log_reg_final_predict)
log_reg_final_f1 = f1_score(y_test, log_reg_final_predict, average='binary')

#print
print('Final Accuracy for Logreg model: ', log_reg_final_accuracy)
print('Final F1 Score for Logreg model: ', log_reg_final_f1)

In [ ]:
# Logistic Regression: print or plot the model coefficients.

#grab coefficients from model
coefficients = log_reg_final_model.coef_[0]

# 10 largest coefficients by magnitude
top_10 = np.argsort(np.abs(coefficients))[-10:]

# print
for i in top_10:
    print(X_train.columns[i], coefficients[i])

# plot
plt.figure(figsize=(8, 6))
plt.barh(X_train.columns[top_10], coefficients[top_10])
plt.title('Top 10 Logistic Regression Coefficients')
plt.xlabel('Coefficient Value')
plt.show()

### Neural Network

In [ ]:
# Scale data for the neural network

# Create the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler to transform the test data
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Get the number of features in training data
n_features = X_train_scaled.shape[1]

# Create the neural network model
nn_model = keras.Sequential()

# Create the input layer, add input layer
input_layer = keras.layers.InputLayer(
    shape=(X_train_scaled.shape[1],),
    name="input"
)
nn_model.add(input_layer)

# Create the hidden layers and add the hidden layers to the 'nn_model' object

# hidden layer 1
hidden_layer_1 = keras.layers.Dense(units=64, activation='relu', name='hl_1')
nn_model.add(hidden_layer_1)

# hidden layer 2
hidden_layer_2 = keras.layers.Dense(units=32, activation='relu', name='hl_2')
nn_model.add(hidden_layer_2)

#hidden layer 3
hidden_layer_3 = keras.layers.Dense(units=16, activation='relu', name='hl_3')
nn_model.add(hidden_layer_3)

# Create the output layer and add the output layer to the 'nn_model' object
output_layer = keras.layers.Dense(units=1, activation='sigmoid', name='output')
nn_model.add(output_layer)

# Print a summary of your model
nn_model.summary()

In [ ]:
sgd_optimizer = keras.optimizers.SGD(learning_rate=0.01) 

In [ ]:
loss_fn = keras.losses.BinaryCrossentropy(from_logits=False)

In [ ]:
nn_model.compile(optimizer=sgd_optimizer, loss=loss_fn, metrics=['accuracy'])

In [ ]:
class ProgBarLoggerNEpochs(keras.callbacks.Callback):
    
    def __init__(self, num_epochs: int, every_n: int = 50):
        self.num_epochs = num_epochs
        self.every_n = every_n
    
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            s = 'Epoch [{}/ {}]'.format(epoch + 1, self.num_epochs)
            logs_s = ['{}: {:.4f}'.format(k.capitalize(), v)
                      for k, v in logs.items()]
            s_list = [s] + logs_s
            print(', '.join(s_list))


In [ ]:
t0 = time.time() # start time

num_epochs = 60

history = nn_model.fit(
    X_train_scaled,
    y_train,
    epochs=num_epochs,
    verbose=0,
    callbacks=[ProgBarLoggerNEpochs(num_epochs, every_n=5)],
    validation_split=0.2
)

t1 = time.time() # stop time

print('Elapsed time: %.2fs' % (t1-t0))

In [ ]:
# Plot training loss and validation loss over epochs

# plot 1
plt.plot(range(1, num_epochs + 1), history.history['loss'], label='Training Loss')
plt.plot(range(1, num_epochs + 1), history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot training accuracy and validation accuracy over epochs

# plot 2
plt.plot(range(1, num_epochs + 1), history.history['accuracy'], label='Training Accuracy')
plt.plot(range(1, num_epochs + 1), history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Generate predictions from neural network using scaled test data
# apply a threshold of 0.5 to get class labels

nn_prob_predictions = nn_model.predict(X_test_scaled)

# 0.5 threshold
for i in range(0,10):
    if nn_prob_predictions[i] >= 0.5:
        class_pred = 1
    else:
        class_pred = 0
    print(str(nn_prob_predictions[i]) + "\t\t\t" + str(class_pred))

# save
nn_class_predictions = (nn_prob_predictions >= 0.5).astype(int)

In [ ]:
# Compute accuracy and F1 score for the neural network and print results
nn_accuracy = accuracy_score(y_test, nn_class_predictions)
nn_f1 = f1_score(y_test, nn_class_predictions, average='binary')

print('Neural Network Accuracy: ', nn_accuracy)
print('Neural Network F1 Score: ', nn_f1)

###  Results Summary

In [ ]:
# side-by-side comparison of two models using the metric variables

results = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score'],
    'Logistic Regression ': [log_reg_final_accuracy, log_reg_final_f1],
    'Neural Network': [nn_accuracy, nn_f1]
})

results